In [8]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F 
from pyspark.sql.types import (StructType, StructField, IntegerType, StringType)

spark = SparkSession.builder.appName("bdpa-clickflow-eclothing-recommender").config('spark.sql.shuffle.partitions', "50").getOrCreate()


raw = (
    spark.read
        .option("sep", ";")
        .option("header", "true")
        .csv("../dataset/e_shop_clothing_2008.csv")
        .toDF(
              "year", "month", "day", "order", "country",
        "session_id", "category", "item_id", "colour",
        "location", "photo_type", "price", "price_above_avg", "page"
        )
        .withColumn("year",            F.col("year").cast(IntegerType()))
        .withColumn("month",           F.col("month").cast(IntegerType()))
        .withColumn("day",             F.col("day").cast(IntegerType()))
        .withColumn("order",           F.col("order").cast(IntegerType()))
        .withColumn("country",         F.col("country").cast(IntegerType()))
        .withColumn("session_id",      F.col("session_id").cast(IntegerType()))
        .withColumn("category",        F.col("category").cast(IntegerType()))
        .withColumn("colour",          F.col("colour").cast(IntegerType()))
        .withColumn("location",        F.col("location").cast(IntegerType()))
        .withColumn("photo_type",      F.col("photo_type").cast(IntegerType()))
        .withColumn("price",           F.col("price").cast(IntegerType()))
        .withColumn("price_above_avg", F.col("price_above_avg").cast(IntegerType()))
        .withColumn("page",            F.col("page").cast(IntegerType()))
) 
    

assert raw.count() == 165474
assert raw.filter(F.col("session_id").isNull()).count() == 0
assert raw.filter(F.col("item_id").isNull()).count() == 0
assert raw.select("month").distinct().count() == 5 # april to august
assert raw.select("category").distinct().count() == 4 # trousers, skirts, blouses, sale
assert raw.select("item_id").distinct().count() == 217


print("All validation checks passed.")
raw.printSchema()
raw.show(5)

        


All validation checks passed.
root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- order: integer (nullable = true)
 |-- country: integer (nullable = true)
 |-- session_id: integer (nullable = true)
 |-- category: integer (nullable = true)
 |-- item_id: string (nullable = true)
 |-- colour: integer (nullable = true)
 |-- location: integer (nullable = true)
 |-- photo_type: integer (nullable = true)
 |-- price: integer (nullable = true)
 |-- price_above_avg: integer (nullable = true)
 |-- page: integer (nullable = true)

+----+-----+---+-----+-------+----------+--------+-------+------+--------+----------+-----+---------------+----+
|year|month|day|order|country|session_id|category|item_id|colour|location|photo_type|price|price_above_avg|page|
+----+-----+---+-----+-------+----------+--------+-------+------+--------+----------+-----+---------------+----+
|2008|    4|  1|    1|     29|         1|       1|    A13|     1|  

In [9]:
session_lengths = (
    raw.groupBy("session_id")
        .agg(F.count("*").alias("n_clicks"))
)

session_lengths.describe("n_clicks").show()

n_single = session_lengths.filter(F.col("n_clicks") == 1).count()

n_total = session_lengths.count() 
print(f"Single click session: {n_single}/{n_total} ({(n_single/n_total * 100)} %)")


# item popularity

item_freq = (
    raw.groupBy('item_id') 
        .agg(F.count("*").alias("views"))
        .orderBy(F.desc("views"))
)

item_freq.show(10)


n_sessions = raw.select("session_id").distinct().count() 
n_items = raw.select("item_id").distinct().count() 
n_pairs = raw.select("session_id", "item_id").distinct().count() 
sparsity = 1 - n_pairs / (n_sessions * n_items)

print(f"Utility matrix: {n_sessions} x {n_items}")
print(f"Observed pairs: {n_pairs}")
print(f"Sparsity: {sparsity:.4%}")


# temporal distribution 
raw.groupBy("month").count().orderBy("month").show() 

raw.groupBy("category").count().orderBy("category").show()

(
    raw.groupBy("session_id", "country")
    .count() 
    .groupBy("country")
    .agg(F.countDistinct("session_id").alias("n_sessions"))
    .orderBy(F.desc("n_sessions"))
    .show(10)
)




+-------+------------------+
|summary|          n_clicks|
+-------+------------------+
|  count|             24026|
|   mean|6.8872887704986265|
| stddev| 8.995160632067192|
|    min|                 1|
|    max|               195|
+-------+------------------+

Single click session: 5042/24026 (20.985598934487637 %)
+-------+-----+
|item_id|views|
+-------+-----+
|     B4| 3579|
|     A2| 3013|
|    A11| 2789|
|     P1| 2681|
|    B10| 2566|
|     A4| 2522|
|    A15| 2489|
|     A5| 2354|
|    A10| 2280|
|     A1| 2265|
+-------+-----+
only showing top 10 rows
Utility matrix: 24026 x 217
Observed pairs: 148345
Sparsity: 97.1547%
+-----+-----+
|month|count|
+-----+-----+
|    4|48199|
|    5|35654|
|    6|32242|
|    7|35231|
|    8|14148|
+-----+-----+

+--------+-----+
|category|count|
+--------+-----+
|       1|49742|
|       2|38408|
|       3|38577|
|       4|38747|
+--------+-----+

+-------+----------+
|country|n_sessions|
+-------+----------+
|     29|     19582|
|      9|      

In [10]:
# implicit interaction matrix 
# apply Hu-Koren-Volinsky confidence weighting scheme


ALPHA = 40.0

interactions = (
    raw.groupBy("session_id", "item_id") 
    .agg(
        F.count("*").alias("click_count"),
        F.min("order").alias("first_click_order"), # earliest position in session
        F.max("month").alias("last_month"),
    ).withColumn("preference", F.lit(1)) # binary indicator
    .withColumn("confidence", 
                F.lit(1.0) + F.lit(ALPHA) * F.col("click_count"))
)

interactions.describe("click_count", "confidence").show() 
interactions.show(5)

+-------+------------------+-----------------+
|summary|       click_count|       confidence|
+-------+------------------+-----------------+
|  count|            148345|           148345|
|   mean|1.1154673227948364|45.61869291179345|
| stddev|0.4505651492305882|18.02260596922328|
|    min|                 1|             41.0|
|    max|                16|            641.0|
+-------+------------------+-----------------+

+----------+-------+-----------+-----------------+----------+----------+----------+
|session_id|item_id|click_count|first_click_order|last_month|preference|confidence|
+----------+-------+-----------+-----------------+----------+----------+----------+
|         2|     P1|          1|                8|         4|         1|      41.0|
|         3|    B17|          1|                1|         4|         1|      41.0|
|        12|    B30|          1|                4|         4|         1|      41.0|
|        13|     A2|          1|                2|         4|         1|

In [11]:
test_sessions = (
    raw.filter(F.col("month") == 8)
        .select("session_id")
        .distinct()
)

# sessions that appeared before august
train_sessions = (
    raw.select("session_id").distinct() 
        .join(test_sessions, on='session_id', how='left_anti')
)

train = interactions.join(train_sessions, on='session_id', how="inner")
test = interactions.join(test_sessions, on='session_id', how="inner")


# ground truth: items clicked by each test session 

ground_truth = (
    raw.filter(F.col("month") == 8)
    .select("session_id", "item_id") 
    .distinct()
)

print(f"Train sessions: {train.select('session_id').distinct().count()}")
print(f"Test sessions:  {test.select('session_id').distinct().count()}")
print(f"Ground truth pairs: {ground_truth.count()}")


Train sessions: 22102
Test sessions:  1924
Ground truth pairs: 12772


In [13]:
# minimum interaction threshold = at least 2 distinct items per session in training

valid_train_sessions = (
    train.groupBy("session_id")
         .agg(F.countDistinct("item_id").alias("n_distinct_items"))
         .filter(F.col("n_distinct_items") >= 2)
         .select("session_id")
)

train_filtered = train.join(valid_train_sessions, on="session_id", how="inner")

removed = train.select("session_id").distinct().count() \
         - train_filtered.select("session_id").distinct().count()
print(f"Sessions removed by threshold: {removed}")
print(f"Training sessions retained: {train_filtered.select('session_id').distinct().count()}")

Sessions removed by threshold: 5058
Training sessions retained: 17044
